# Quantitative metrics

In "The Tale of the Deep Learning Model That Failed My Driving Exam", we left the driving evaluator puzzled. He watched as the deep learning model correctly stopped at the intersection with a yellow light and an ambulance crossing. However, when he asked the model to explain how it came to such a decision, he realized the model’s responses were purely technical, focused on pixel values and activations rather than a true understanding of traffic rules or emergency vehicles.

To help the driving evaluator, in the last chapter, we introduced **Concept Bottleneck Models (CBMs)**. CBMs allow the evaluator to gain insights into the model's decision-making process as:
*    **Task predictions can be traced back to the activation of human-interpretable concepts**. This enables the model to answer the driving evaluator's question by saying, "I decided to cross as I saw a green light and there was no ambulance".
*    **Altering concept values changes the model's decisions**. This allows the model to respond to the evaluator's question with, "In the same scenario, if there is an ambulance, I would cross".

While these qualitative answers reassured the evaluator, he now needs a **quantitative** way to assess how well the model responds in different situations. For this, we use three key metrics in concept-based interpretability: concept/task **predictive performance**, **intervention effectiveness**, and **concept completeness**.

In the following, we assume we have a CBM $(\theta_g, \theta_f)$ and a dataset of i.i.d. triples (input, concepts, task) $\mathcal{D} = \{(\hat{x}, \hat{c}, \hat{y})\}$, which we use to compute these metrics.


## 1. Predictive performance

### Task performance  
In the driving test, task performance {cite}`koh2020concept` measures how well the model predicts whether to cross or stop at the intersection, based on concepts like traffic light color and the ambulance. A high task performance indicates that the model generally makes the correct decision. Task performance is usually represented as the likelihood:


$$\mathcal{L}(\theta_f, \theta_g, \mathcal{D}) = \frac{1}{|\mathcal{D}|} \sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \ p(\hat{y} \mid g(\hat{x}); \theta_f)$$

**Limitations:** High task performance alone doesn’t guarantee interpretability. The model might achieve high performance by relying on uninterpretable or poorly defined concepts, potentially missing key information. For example, a CBM could correctly predict whether to cross or stop, but its decisions might be unaffected by changing the value of the concept “ambulance".


### Concept performance  
Concept performance {cite}`koh2020concept` measures how well the model correctly predicts the concepts from the input data. In our driving example, this evaluates how well the model detects traffic light color or the ambulance’s presence. Similarly to task performance, concept performance can be measured as the likelihood:

$$\mathcal{L}(\theta_g, \mathcal{D}) = \frac{1}{|\mathcal{D}|} \sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \ p(\hat{c} \mid \hat{x}; \theta_g)$$

**Limitations:** While high concept performance means the model can correctly identify the traffic light color and ambulance presence, it doesn’t always imply that the model will make the right final decision. For example, even if the model accurately identifies a green light and no ambulance, it might still incorrectly predict that the car should stop.

## 2. Intervention effectiveness

In the coding practice of the previous chapter, we observed how changing the predicted value of a concept affects task predictions. This property allows human experts to intervene on mispredicted concept values and correct them to improve the model's task performance. Intervention effectiveness {cite}`koh2020concept` measures how well a CBM improves task performance after such corrections. This can be achieved by computing the area under the intervention curve as follows:
* consider a subset of the powerset of the concept indices $\mathcal{I} \subseteq \mathcal{P}(\{1, \dots, |\mathcal{C}|\})$ to intervene on;
* for each concept group $\mathcal{I}' \in \mathcal{I}$, replace the subset of predictions $g(\hat{x})_{\mathcal{I}'}$ with ground truth values $\hat{c}_{\mathcal{I}'}$;

$$\mathcal{Q}(\theta_f, \theta_g, \mathcal{D}, \mathcal{I}) = \frac{1}{|\mathcal{D}| \cdot |\mathcal{I}|} \sum_{\mathcal{I}' \in \mathcal{I}} \sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}}  \ p(\hat{y} \mid g(\hat{x})_{\mathcal{I}'} = \hat{c}_{\mathcal{I}'}; \theta_f)$$

In the driving test scenario, imagine the model incorrectly predicts that the ambulance is absent, leading to a wrong decision to cross. This metric would evaluate whether correcting the concept by setting "ambulance = 1" leads the model to change its prediction to the correct one (i.e., stopping). After this human intervention, we expect the model to revise its decision and recommend stopping the car.

Intervention effectiveness is particularly useful when human experts need to intervene and adjust concept values to improve the model’s decisions, ensuring the model responds accurately after corrections.

**Limitations:** This metric is highly dependent on the quality of the interventions provided by humans. Poorly chosen interventions can reduce the model's performance, even if the concepts themselves are correct.

## 3. Concept completeness

In the driving test example, the driving evaluator may wonder whether a CBM underperforms compared to a standard black-box model that does not use concepts. This question can be answered using concept completeness {cite}`yeh2020completeness`, a metric which helps determine if the concepts alone provide sufficient information for the model to solve the task when compared to a black box. Concept completeness<sup>1</sup> is defined as the ratio of the task performance of a CBM to that of a black-box model $(\theta_b)$:

$$\mathcal{B}(\theta_f, \theta_g, \theta_b, \mathcal{D}) = \frac{\sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \ p(\hat{y} \mid g(\hat{x}); \theta_f)}{\sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \ p(\hat{y} \mid \hat{x}, \hat{c}); \theta_b)}$$

In the driving test scenario, if the CBM uses concepts like “traffic light color” and “ambulance presence” and achieves a task accuracy that is close to or greater than a black-box model trained directly on raw input (e.g., image pixels), it implies that the concepts provide the necessary information to complete the task. For instance, if the CBM’s accuracy in predicting whether to stop or cross is 95% and the black-box model’s accuracy is 96%, the high concept completeness score shows that interpretability is achieved with minimal loss in task accuracy.

**Limitations:** High concept completeness does not guarantee that the concepts themselves are fully interpretable or well-defined; a model might achieve high completeness while relying on concepts that are difficult for humans to understand.

## Coding practice: quantitative metrics for concept-based models

In this practice, we will evaluate a CBM's predictive performance, intervention effectiveness, and concept completeness in a simple traffic light scenario where decisions to cross or stop depend on two concepts: the traffic light being green and the presence of an ambulance. The model predicts these concepts and uses them to make decisions.

> ⚠️ **Note:** This section assumes familiarity with the first chapter on CBMs and PyC. If you're new to these concepts, it's recommended to start with [this introductory chapter](https://link_to_chapter) for foundational understanding.

### Step #1: Train a CBM

We begin this practice by re-training the Concept Bottleneck Model (CBM) used in the previous coding exercise.

In [1]:
%%capture
!pip install torch
!pip install torch_geometric
!pip install -i https://test.pypi.org/simple/ --upgrade pytorch-concepts

import torch
import numpy as np
from torch_concepts.nn import ConceptEncoder
from torch_concepts.data import TrafficLights
from sklearn.model_selection import train_test_split

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

n_samples = 1000
latent_dims = 5
n_epochs = 200
concept_reg = 0.5
lr = 0.01

# Loading dataset
dataset = TrafficLights(n_samples=n_samples)
x, c, y, concept_names, task_names = dataset.x_train, dataset.c_train, dataset.y_train, dataset.concept_names, dataset.task_names
x_train, x_test, c_train, c_test, y_train, y_test = train_test_split(x, c, y, test_size=0.2, random_state=42)

# Defining the CBM
# The encoder extracts a low-dimensional representation of the input
encoder = torch.nn.Sequential(
    torch.nn.Linear(x_train.shape[1], latent_dims),
    torch.nn.LeakyReLU()
)
# The concept scorer predicts concept logits {traffic light color, ambulance presence}
c_scorer = ConceptEncoder(
    in_features=latent_dims,
    out_concept_dimensions={1: concept_names}
)
# The task predictor determines the value of the downstream label {cross}
y_predictor = torch.nn.Sequential(
    torch.nn.Linear(c_train.shape[1], latent_dims),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(latent_dims, y_train.shape[1])
)
model = torch.nn.Sequential(encoder, c_scorer, y_predictor)

# Define optimizer and loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
loss_fn = torch.nn.BCELoss()

# Standard PyTorch learning cycle
model.train()
for epoch in range(n_epochs):
   optimizer.zero_grad()

   # Encode input, then predict concept and downstream tasks activations
   emb = encoder(x_train)
   c_pred = c_scorer(emb).sigmoid()
   y_pred = y_predictor(c_pred).sigmoid()

   # Double loss on concepts and tasks
   loss = loss_fn(c_pred, c_train) + concept_reg * loss_fn(y_pred, y_train)
   loss.backward()
   optimizer.step()

model.eval()
c_pred = c_scorer(encoder(x_test)).sigmoid()
y_pred = y_predictor(c_pred).sigmoid()


### Step #2: Compute task and concept performance

In our traffic light dataset, both concepts ("traffic light color" and "ambulance presence") and the output ("cross") are binary variables. This allows us to evaluate task and concept performance using the Area Under the Receiver Operating Characteristic Curve (ROC AUC) from prediction scores, a standard metric for binary classification problems.

In [2]:
from sklearn.metrics import roc_auc_score

concept_performance = roc_auc_score(c_test, c_pred.detach())
task_performance = roc_auc_score(y_test, y_pred.detach())

print(f'Task performance: {task_performance:.4f}')
print(f'Concept performance: {concept_performance:.4f}')

Task performance: 0.9213
Concept performance: 0.9443


After a few epochs, the CBM is already able to accurately predict both concepts and tasks.


### Step #3: Compute intervention effectiveness

In our traffic light scenario, we can evaluate how intervening by correcting mispredicted concepts—such as "traffic light color" or "ambulance presence"—improves the model’s decision on whether to "cross." By replacing predicted concept values with ground truth values and recalculating model performance, we measure how each intervention impacts downstream task accuracy. In PyC, this requires specifying a list of concept index groups `I` to intervene on.


In [3]:
from torch_concepts.metrics import intervention_score

intervention_groups = [[], [0], [1], [0, 1]]

# Evaluate intervention effectiveness of each concept group individually
intervention_scores = intervention_score(y_predictor, c_pred,
                                         c_test, y_test, intervention_groups,
                                         auc=False)
print(f'Individual intervention scores: {intervention_scores}')

# Evaluate the global intervention effectiveness as the AUC
intervention_auc = intervention_score(y_predictor, c_pred,
                                      c_test, y_test, intervention_groups)
print(f'Intervention AUC: {intervention_auc:.4f}')

Individual intervention scores: [0.9212905020164093, 0.9937421777221527, 0.9585593102489223, 1.0]
Intervention AUC: 0.9684


The high intervention AUC indicates that concept interventions effectively improve the model’s task performance. Individual intervention scores show that correcting the "traffic light color" concept provides the most significant improvement in downstream task performance.


### Step #4: Compute concept completeness

To compute concept completeness, we need a black-box baseline model that uses both raw features and concept labels, as this matches the information provided to the CBM. The following code implements a simple black-box model with a similar parameter count to the CBM for a fair comparison.

In [4]:
xc_train = torch.cat((x_train, c_train), dim=1)
xc_test = torch.cat((x_test, c_test), dim=1)

# Defining a balck box baseline
baseline = torch.nn.Sequential(
    torch.nn.Linear(xc_train.shape[1], latent_dims),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(latent_dims, latent_dims),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(latent_dims, latent_dims),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(latent_dims, y_train.shape[1])
)

# Define optimizer and loss function
optimizer = torch.optim.AdamW(baseline.parameters(), lr=lr)
loss_fn = torch.nn.BCELoss()

# Standard PyTorch learning cycle
baseline.train()
for epoch in range(n_epochs):
   optimizer.zero_grad()

   # Encode input, then predict concept and downstream tasks activations
   y_pred_baseline = baseline(xc_train).sigmoid()

   # Double loss on concepts and tasks
   loss = loss_fn(y_pred_baseline, y_train)
   loss.backward()
   optimizer.step()

baseline.eval()
y_pred_baseline = baseline(xc_test).sigmoid()
task_performance_baseline = roc_auc_score(y_test, y_pred_baseline.detach())

We can then compute concept completeness as the ratio between the CBM’s downstream task performance and that of the black-box baseline. This ratio indicates the cost of incorporating a concept layer within the network to improve interpretability compared to an equivalent black-box model.

In [5]:
from torch_concepts.metrics import completeness_score

concept_completeness = completeness_score(y_test, y_pred_baseline, y_pred)

print(f'Task performance: {task_performance:.4f}')
print(f'Task performance baseline: {task_performance_baseline:.4f}')
print(f'Concept completeness: {concept_completeness:.4f}')

Task performance: 0.9213
Task performance baseline: 1.0000
Concept completeness: 0.9213


The resulting concept completeness score suggests that the CBM reflects a moderate drop in downstream task performance compared to the black-box baseline, in exchange for the increased interpretability.

## Take Home Message

In summary, this chapter demonstrated how to quantitatively evaluate CBMs using metrics such as task and concept performance, intervention effectiveness, and concept completeness. These metrics provide a insight into the model’s accuracy and interpretability as follows:

- **Task and concept performance:** This metric quantifies a CBM's predictive performance, assessing how accurately the model predicts concepts and downstream tasks.
- **Intervention effectiveness:** This metric quantifies the impact of concepts on a CBM's task predictions. Specifically, concept interventions can be used to assess how correcting mispredicted concepts can improve downstream task performance.
- **Concept completeness:** This metric quantifies the cost in downstream task performance of introducing a concept layer compared to a black-box model.



## Next chapter

The next chapter will discuss the issue of concept leakage, a potential challenge in concept-based interpretability, and strategies to mitigate its impact on CBMs.


## References

Koh, Pang Wei, et al. "Concept bottleneck models." International conference on machine learning. PMLR, 2020.

Yeh, Chih-Kuan, et al. "On completeness-aware concept-based explanations in deep neural networks." Advances in neural information processing systems 33 (2020): 20554-20565.[testo del link](https:// [testo del link](https://))



--------------------
<sup>1</sup>: The formulation provided here is evaluation the *completeness of the model*. Equivalently, we can compute the *completeness of the data* by replacing concept predictions with concept truth values in the numerator:

$$\mathcal{B}(\theta_f, \theta_g, \theta_b, \mathcal{D}) = \frac{\sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \ p(\hat{y} \mid \hat{c}; \theta_f)}{\sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \ p(\hat{y} \mid \hat{x}, \hat{c}); \theta_b)}$$